# exp00r0 — baseline + noise measurement

**Round** 0 (setup; not an experiment round) · **Parent** none — this *is* the baseline

Round 0 produces the three things every later round depends on, and nothing else:

1. A **clean baseline run** on the upstream config. No changes, no tuning.
2. The platform's **throughput**, which tells you how long each round's runs will
   take so a night's work can be planned (`research-program.md` §1.2).
3. **σ₀** — the seed-to-seed standard deviation of `val_bpb`. Every later keep/discard
   verdict is measured against this. Without it there is no decision rule, only noise.

Do not tune anything here. Round 0 is instrumentation.

## 1 · Setup

`prepare_round()` must run **before** any dataloader or eval call — it is what repoints
`prepare.DATA_DIR` at the round's shard slice. The harness asserts if you forget.

In [ ]:
import sys, os
sys.path.insert(0, "/home/dangnguyenducanh/autoresearch")  # repo root, so `prepare` imports

import prepare

# --- §2 project constants. Pin ONCE for the whole project, never per experiment. ---
# Upstream defaults. Override here only if the platform forces it, and then never again.
# prepare.MAX_SEQ_LEN = 2048
# prepare.EVAL_TOKENS = 40 * 524288

from experiments._shared.data import prepare_round, round_tokens, estimated_runtime_s
from experiments._shared.harness import ExpConfig, run, run_seeds

prepare_round(0)
print(f"MAX_SEQ_LEN={prepare.MAX_SEQ_LEN}  EVAL_TOKENS={prepare.EVAL_TOKENS:,}")

## 2 · Baseline config

Upstream `train.py` values, verbatim. `gpu=` only affects the reported MFU figure, never
`val_bpb` — set it to your card so the MFU number means something.

In [ ]:
baseline = ExpConfig(
    round_idx=0,      # R0 trains on exactly what an R1 run does, so sigma_0 transfers
    seed=0,
    gpu="H100",            # key into harness.PEAK_FLOPS; MFU reporting only
    notes="upstream train.py defaults, unmodified",
)
print(f"model_dim = {baseline.model_dim()}  grad_accum = {baseline.grad_accum_steps()}")
for k, v in vars(baseline).items():
    print(f"  {k:20s} {v}")

## 3 · One run — does it work, and how fast is this machine?

In [ ]:
m = run(baseline, outdir="experiments/exp00r0")
throughput = m["tok_per_s"]
print(f"\nthroughput: {throughput:,} tokens/s")

## 4 · Size the dataset — the epoch gate

Every round must stay under 1.0 epoch. A round that repeats data measures memorization
rather than the change under test, so its verdicts do not transfer to the next round.

If any round comes out ≥ 1.0, **raise `SHARD_BUDGET`** in `experiments/_shared/data.py`
and download more shards. `SHARD_BUDGET` sets the project's absolute compute scale:
every round is a fraction of it, and a run is one pass over that fraction.

In [ ]:
print(f"epoch reached: {m['epoch']:.3f}  (one pass by design, ~1.0 expected)\n")

print("Planned work, at this machine's throughput:")
total = 0
for r, runs in [(1, 25), (2, 12), (3, 5)]:
    secs = estimated_runtime_s(r, throughput)
    total += secs * runs
    print(f"  round {r}: {round_tokens(r)/1e9:5.2f}B tokens/run  "
          f"{secs/60:5.1f} min/run  x ~{runs} runs = {secs*runs/3600:5.1f} h")
print(f"  total: {total/3600:.1f} h of GPU time for the whole project")
print("\nRaise or lower SHARD_BUDGET in experiments/_shared/data.py to scale this.")

## 5 · σ₀ — the noise band

Three runs, identical config, seeds 0/1/2. The spread is the resolution limit of this
whole project: any `val_bpb` difference smaller than ~2σ is not a result.

Record σ₀ in `rounds/r1-directions.md`. Round 1 cannot open without it.

In [ ]:
mean0, sigma0, runs0 = run_seeds(baseline, seeds=(0, 1, 2), outdir="experiments/exp00r0")
print(f"\nBASELINE  mean val_bpb = {mean0:.6f}")
print(f"SIGMA_0   = {sigma0:.6f}")
print(f"keep threshold for R1 (-2σ): val_bpb < {mean0 - 2 * sigma0:.6f}")
print(f"\nIf sigma0 is 0.000000 the seeds are not actually varying — check that")
print(f"ExpConfig.seed reaches torch.manual_seed before fixing sigma for the round.")

## 6 · R0 exit gate

All three must hold before Round 1 opens (`research-program.md` §1.3):

- [ ] A run completes and prints `val_bpb:` without crashing
- [ ] `epoch` landed near 1.0 (else correct `SHARD_TOKENS`)
- [ ] Runtime per round known, and `SHARD_BUDGET` agreed with the human
- [ ] σ₀ measured from 3 seed repeats

Then: write `experiments/exp00r0/report.md`, agree `rounds/r1-directions.md` with the
human, and **stop**. Round 1 does not start without approval.